# 03 - First Text Models

## Objective

The objective is to obtain the first simple modelling results for the Rakuten product-classification task. Two models are compared:

1. a minimum baseline that always predicts the largest class;
2. a text baseline using TF-IDF and a Linear SVM.

Both models use the existing shared training/validation split. The official test data is not used because its labels are unavailable.

## 1. Load the Prepared Text

The processed file contains cleaned text, the correct product category and the shared split. Product and image identifiers are not used as model features.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC

processed_path = Path("../data/processed/text_train_processed.csv")
data = pd.read_csv(processed_path)

model_train = data[data["split"] == "train"].copy()
model_validation = data[data["split"] == "validation"].copy()

print(f"Training products: {len(model_train):,}")
print(f"Validation products: {len(model_validation):,}")

Training products: 67,932
Validation products: 16,984


## 2. Largest-Class Baseline

This baseline predicts the most common training category for every validation product. It ignores the text completely and establishes a minimum score that a useful model should beat.

In [2]:
y_train = model_train["prdtypecode"]
y_validation = model_validation["prdtypecode"]

largest_class = y_train.value_counts().idxmax()
majority_predictions = np.full(len(model_validation), largest_class)

majority_weighted_f1 = f1_score(
    y_validation, majority_predictions, average="weighted"
)
majority_macro_f1 = f1_score(
    y_validation, majority_predictions, average="macro"
)

print(f"Largest class: {largest_class}")
print(f"Weighted F1: {majority_weighted_f1:.4f}")
print(f"Macro F1: {majority_macro_f1:.4f}")

Largest class: 2583
Weighted F1: 0.0258
Macro F1: 0.0080


## 3. Convert Text into TF-IDF Features

Machine-learning models require numbers rather than sentences. TF-IDF converts the cleaned combined text into numerical word features.

The vocabulary is learned from training text only. The fitted vectorizer then transforms the validation text without learning from it.

In [3]:
vectorizer = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    min_df=2,
)

X_train = vectorizer.fit_transform(model_train["text_combined"])
X_validation = vectorizer.transform(model_validation["text_combined"])

print(f"Training matrix: {X_train.shape}")
print(f"Validation matrix: {X_validation.shape}")

Training matrix: (67932, 50000)
Validation matrix: (16984, 50000)


## 4. Train the First Text Classifier

A Linear SVM is a standard, efficient baseline for high-dimensional TF-IDF text features. It learns from the training products and predicts categories for the unseen validation products.

In [4]:
text_model = LinearSVC(random_state=42)
text_model.fit(X_train, y_train)
text_predictions = text_model.predict(X_validation)

text_weighted_f1 = f1_score(
    y_validation, text_predictions, average="weighted"
)
text_macro_f1 = f1_score(
    y_validation, text_predictions, average="macro"
)

## 5. Compare the Results

Weighted F1 gives more influence to large classes. Macro F1 gives equal importance to every class and therefore reveals whether smaller classes are being neglected.

In [5]:
results = pd.DataFrame({
    "model": ["Largest-class baseline", "TF-IDF + Linear SVM"],
    "weighted_f1": [majority_weighted_f1, text_weighted_f1],
    "macro_f1": [majority_macro_f1, text_macro_f1],
})

results.round(4)

,model,weighted_f1,macro_f1
0,Largest-class baseline,0.0258,0.0080
1,TF-IDF + Linear SVM,0.8274,0.8156


## Conclusion

The largest-class baseline provides the minimum reference score. The TF-IDF model tests whether the cleaned product text contains useful category information. These are initial results, not a final optimized model.

The next text-modelling step is to examine per-class performance and the most frequent classification errors before changing model settings.